# 09 · The planner's levers, and why regions differ

**Levers.** A lever is something a health system can change, written as an edit to each
woman's record among the 21 predictors: a health centre or hospital within an hour, health
insurance, weekly radio health messages (one more weekly media source for a woman who does not
yet listen weekly), every pregnancy wanted, women taking part in decisions on their own
healthcare. Every woman is re-scored by logistic regression with the lever applied, and the
average change in her predicted chance, by region, is the lever's weight (g-computation).
Women already in the better state are unchanged. All combinations of the levers are scored, so
any mix of coverages combines exactly. One more lever, an earlier first visit, uses a model
fitted on the women who started care. Intervals resample whole clusters with the model held
fixed. These are associations: what the model expects if circumstances changed, not what a
programme would cause.

**Why a region differs.** Logistic regression is additive, so the gap between a region's
average predicted log-odds and the country's splits exactly across the 21 predictors, then
sums to planning concepts (access, insurance, wealth, education, and so on).

In [ ]:
# Settings for this notebook
QUICK = False   # True: fewer resamples for the intervals (a check that everything runs)

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib

In [ ]:
repro.run_script("run_scenarios.py", "--models", "logistic", *(["--quick"] if QUICK else []))
repro.run_script("run_regional_eda.py", "--models", "logistic")

In [ ]:
s = json.load(open(P.outputs / "scenarios" / "scenarios.json"))
keys = [l["key"] for l in s["levers"]] + ["all together"]
nat = {r: s["by_model"]["logistic"]["rungs"][r]["gain"][-1] for r in ("anc1", "anc4", "anc8")}
print("Change in the share reaching each rung if every eligible woman is reached (points, Ethiopia):")
print(pd.DataFrame({r: [v * 100 for v in g] for r, g in nat.items()}, index=keys).round(2).to_string())
drv = pd.read_csv(P.outputs / "regional" / "drivers_concepts_by_region.csv")
print("\nWhy Somali's ANC8+ differs from Ethiopia's (log-odds, by concept):")
print(drv[(drv.model == "logistic") & (drv.rung == "anc8") & (drv.region == "Somali")].sort_values("contribution")[["label", "contribution"]].round(3).to_string(index=False))
ref = repro.reference("levers_national.json")["logistic"]
gap = max(abs(a - b) for r in ref for a, b in zip(ref[r], nat[r]))
repro.check("the national lever effects are the study's", gap < 1e-9, f"largest difference {gap:.1e}")
repro.report()